<a href="https://colab.research.google.com/github/Deva2013/airline-disruption-management-system/blob/main/Airline_Disruption_Phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✈️ Airline Disruption Detection & Recovery System — Phase 1

**Status:** Phase 1 complete — cleaned pipeline, no development/debugging artifacts

This notebook covers the full Phase 1 pipeline: BTS flight data (with aircraft
tail number) + IEM historical weather data → disruption detection engine →
feature engineering → XGBoost delay prediction model (ROC-AUC 0.7222) → live
operations dashboard.

**Data & artifacts:** persisted to [Hugging Face Hub](https://huggingface.co/datasets/Dev123Hug456Face/airline-disruption-data)
**Live dashboard:** [phase1_dashboard.html](https://deva2013.github.io/airline-disruption-management-system/phase1_dashboard.html)

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Environment Setup
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Set up local working directories and authenticate with
#          Hugging Face Hub. No Google Drive used anywhere — all
#          working data lives on local Colab disk (/content), and
#          finished outputs are pushed to Hugging Face Hub for
#          permanent storage.
# ═══════════════════════════════════════════════════════════════════════

import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

!pip install -q huggingface_hub fastparquet

from huggingface_hub import HfApi, login, hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

TARGET_AIRPORTS = ['JFK', 'ORD', 'ATL', 'LAX', 'DFW', 'SFO', 'EWR', 'MIA', 'SEA', 'BOS']

print('Environment ready.')

Environment ready.


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Pull Data from Hub, Scope to Origin Hubs, Join with Weather
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Restore the tail-number-enriched BTS dataset and weather
#          data from Hugging Face Hub, scope to origin-hub departures,
#          and join — combining three previous steps into one cell.
# ═══════════════════════════════════════════════════════════════════════

# ── Pull BTS data (with Tail_Number) ────────────────────────────────────
tail_path = hf_hub_download(
    repo_id=HF_REPO_ID, filename="bts_cleaned_with_tail.parquet",
    repo_type="dataset", local_dir=DIR_PROCESSED,
)
df_tail = pd.read_parquet(tail_path)
print(f"BTS data: {len(df_tail):,} rows")

# ── Scope to origin-hub departures only ─────────────────────────────────
df_scoped = df_tail[df_tail['Origin'].isin(TARGET_AIRPORTS)].copy()
del df_tail
print(f"Scoped to origin hubs: {len(df_scoped):,} rows")

# ── Pull weather data ────────────────────────────────────────────────────
wx_path = hf_hub_download(
    repo_id=HF_REPO_ID, filename="metar_clean.parquet",
    repo_type="dataset", local_dir=DIR_WEATHER,
)
wx = pd.read_parquet(wx_path)
print(f"Weather data: {len(wx):,} rows")

# ── Join ──────────────────────────────────────────────────────────────
df_scoped['JoinDate'] = df_scoped['FlightDate'].dt.date
bts_weather = df_scoped.merge(
    wx, left_on=['Origin', 'JoinDate', 'ScheduledDepHour'],
    right_on=['IATA', 'Date', 'DepHour'], how='left'
)
bts_weather = bts_weather.drop(columns=['JoinDate', 'IATA', 'Date', 'DepHour'])
del df_scoped

matched = bts_weather['FlightCategory'].notna().sum()
print(f'\nJoined: {len(bts_weather):,} rows, {matched/len(bts_weather)*100:.2f}% weather match')

BTS data: 10,504,936 rows
Scoped to origin hubs: 5,896,606 rows
Weather data: 263,034 rows

Joined: 5,896,606 rows, 100.00% weather match


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Reduce Memory Footprint Before Feature Engineering
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Convert repetitive string/object columns to 'category' dtype
#          before doing any further feature engineering. This is the
#          step we missed in the previous rebuild attempt — going
#          straight from join into feature engineering (with the added
#          weight of the high-cardinality Tail_Number column) is what
#          caused the RAM crash.
# ═══════════════════════════════════════════════════════════════════════

import gc

category_cols = [
    'Airline', 'OperatingAirline', 'OperatingAirlineCode',
    'Origin', 'OriginCityName', 'OriginState',
    'Dest', 'DestCityName', 'DestState',
    'CancellationReason', 'PrimaryDelayCause', 'SeverityTier', 'Season',
    'FlightCategory',
]

for col in category_cols:
    if col in bts_weather.columns:
        bts_weather[col] = bts_weather[col].astype('category')

# NOTE: Tail_Number is NOT converted to category — it has very high
# cardinality (thousands of unique aircraft), so category dtype would
# add overhead rather than save memory. It stays as a string column,
# and we'll only use it briefly for grouping/sorting in the next step.

gc.collect()

print('Memory optimization complete.')
print(f'bts_weather memory usage: {bts_weather.memory_usage(deep=True).sum() / 1e9:.2f} GB')
!free -h

Memory optimization complete.
bts_weather memory usage: 1.54 GB
               total        used        free      shared  buff/cache   available
Mem:            12Gi       5.4Gi       5.4Gi       2.0Mi       1.9Gi       7.0Gi
Swap:             0B          0B          0B


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Disruption Detection Engine (Vectorized)
# ═══════════════════════════════════════════════════════════════════════
bts_weather['WeatherContributed'] = (
    (bts_weather['IsLowVis'] == 1) |
    (bts_weather['IsHighWind'] == 1) |
    (bts_weather['IsGust'] == 1) |
    (bts_weather['IsFogOrIFR'] == 1)
).astype(int)

conditions = [
    bts_weather['Cancelled'] == 1,
    bts_weather['Diverted'] == 1,
    bts_weather['ArrDelayMinutes'] < 15,
    bts_weather['WeatherContributed'] == 1,
    bts_weather['PrimaryDelayCause'].notna(),
]

choices = [
    'Cancelled',
    'Diverted',
    'On Time',
    'Weather-Related Delay',
    bts_weather['PrimaryDelayCause'].astype(str) + ' Delay',
]

bts_weather['DisruptionType'] = np.select(conditions, choices, default='Other Delay')
bts_weather['IsCascadeRisk'] = (bts_weather['LateAircraftDelay'] > 0).astype(int)

print('Disruption Type breakdown:')
print(bts_weather['DisruptionType'].value_counts().to_string())

Disruption Type breakdown:
DisruptionType
On Time                  4525651
Carrier Delay             432795
Late Aircraft Delay       392168
NAS Delay                 288989
Cancelled                 104301
Weather-Related Delay      99051
Weather Delay              36421
Diverted                   14775
Security Delay              2453
Other Delay                    2


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Congestion and Rolling Delay-Rate Features
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Rebuild the two engineered features validated earlier —
#          AirportHourCongestion (flight count per airport-hour) and
#          RollingDelayRate (prior-day airline+airport delay rate,
#          lagged to avoid leakage).
# ═══════════════════════════════════════════════════════════════════════

# ── Feature 1: Airport-hour congestion ──────────────────────────────────
bts_weather['AirportHourCongestion'] = bts_weather.groupby(
    ['Origin', 'FlightDate', 'ScheduledDepHour'], observed=True
)['Origin'].transform('count')

# ── Feature 2: Rolling 24-hour delay rate (per airline + airport) ──────
bts_weather = bts_weather.sort_values(['Airline', 'Origin', 'FlightDate', 'ScheduledDepHour'])

daily_rate = bts_weather.groupby(
    ['Airline', 'Origin', 'FlightDate'], observed=True
)['DepDel15'].mean().reset_index()
daily_rate = daily_rate.rename(columns={'DepDel15': 'DailyDelayRate'})

daily_rate = daily_rate.sort_values(['Airline', 'Origin', 'FlightDate'])
daily_rate['RollingDelayRate'] = daily_rate.groupby(
    ['Airline', 'Origin'], observed=True
)['DailyDelayRate'].shift(1)

bts_weather = bts_weather.merge(
    daily_rate[['Airline', 'Origin', 'FlightDate', 'RollingDelayRate']],
    on=['Airline', 'Origin', 'FlightDate'],
    how='left'
)

overall_avg = bts_weather['DepDel15'].mean()
bts_weather['RollingDelayRate'] = bts_weather['RollingDelayRate'].fillna(overall_avg)

gc.collect()

print('AirportHourCongestion — mean:', bts_weather['AirportHourCongestion'].mean().round(2),
      ' min:', bts_weather['AirportHourCongestion'].min(),
      ' max:', bts_weather['AirportHourCongestion'].max())
print('RollingDelayRate — mean:', bts_weather['RollingDelayRate'].mean().round(4),
      ' min:', bts_weather['RollingDelayRate'].min(),
      ' max:', bts_weather['RollingDelayRate'].max())

AirportHourCongestion — mean: 40.43  min: 1  max: 108
RollingDelayRate — mean: 0.209  min: 0.0  max: 1.0


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Build Aircraft-Chain Delay Feature (Tail_Number)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: For each flight, determine whether that SPECIFIC aircraft's
#          previous leg (by Tail_Number) arrived late. This is a
#          genuinely different signal from RollingDelayRate (which is
#          airline+airport level) — it tracks the individual plane's
#          own rotation through the day.
#
# Important caveat: our dataset is scoped to flights touching the 10
# hub airports, so an aircraft's "previous flight" in this dataset may
# not be its true previous flight in the full national network (if the
# prior leg didn't touch a hub, it's not in our data). This is a known
# limitation — the feature is a defensible on the departures we *did*
# capture at hub airports, but shift(1) will sometimes point to a leg
# that is not physically the immediately prior one.
# ═══════════════════════════════════════════════════════════════════════

# Build a proper chronological ordering key from date + scheduled dep time
bts_weather['DepDateTime'] = pd.to_datetime(
    bts_weather['FlightDate'].astype(str) + ' ' +
    bts_weather['CRSDepTime'].astype(str).str.zfill(4).str[:2] + ':' +
    bts_weather['CRSDepTime'].astype(str).str.zfill(4).str[2:],
    errors='coerce'
)

# Sort by aircraft + chronological order — required before using shift()
bts_weather = bts_weather.sort_values(['Tail_Number', 'DepDateTime'])

# For each aircraft, get the PREVIOUS flight's arrival delay (in minutes)
# and whether it was delayed — shift(1) looks at the row immediately
# before in this sorted order, which for a given Tail_Number is its
# prior leg within our dataset.
bts_weather['PrevLegArrDelay'] = bts_weather.groupby(
    'Tail_Number', observed=True
)['ArrDelayMinutes'].shift(1)

bts_weather['PrevLegWasDelayed'] = bts_weather.groupby(
    'Tail_Number', observed=True
)['ArrDel15'].shift(1)

# Missing values occur for: (a) an aircraft's FIRST flight in our
# dataset (no prior leg to reference), or (b) missing Tail_Number.
# Fill with 0 (treat as "not known to be delayed") — a reasonable,
# conservative default.
bts_weather['PrevLegArrDelay'] = bts_weather['PrevLegArrDelay'].fillna(0)
bts_weather['PrevLegWasDelayed'] = bts_weather['PrevLegWasDelayed'].fillna(0).astype(int)

gc.collect()

print(f'PrevLegWasDelayed — rate: {bts_weather["PrevLegWasDelayed"].mean()*100:.2f}%')
print(f'PrevLegArrDelay — mean: {bts_weather["PrevLegArrDelay"].mean():.2f} min, '
      f'max: {bts_weather["PrevLegArrDelay"].max():.0f} min')
print(f'\nRows with a known previous leg: '
      f'{(bts_weather["PrevLegArrDelay"] != 0).sum() + (bts_weather["PrevLegWasDelayed"] == 0).sum() - len(bts_weather) + (bts_weather["PrevLegArrDelay"] == 0).sum():,}')

PrevLegWasDelayed — rate: 21.21%
PrevLegArrDelay — mean: 15.54 min, max: 5986 min

Rows with a known previous leg: 4,646,229


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Save Final Feature Set Locally, Then Upload to Hub
# ═══════════════════════════════════════════════════════════════════════
final_path = DIR_PROCESSED / 'bts_features_with_tail.parquet'
bts_weather.to_parquet(final_path, index=False)
print(f'Saved: {final_path}')

hf_api.upload_file(
    path_or_fileobj=str(final_path),
    path_in_repo="bts_features_with_tail.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print("✅ Uploaded to Hugging Face Hub")

Saved: /content/airline-disruption/data/processed/bts_features_with_tail.parquet


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eatures_with_tail.parquet:   0%|          |  526kB /  227MB            

✅ Uploaded to Hugging Face Hub


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Retrain XGBoost with Aircraft-Chain Delay Features
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Same setup as before (time-based split, class imbalance
#          handling), now adding PrevLegWasDelayed and PrevLegArrDelay
#          — testing whether the aircraft-chain signal beats the
#          0.6903 ROC-AUC ceiling from the airline/airport-level
#          RollingDelayRate alone.
# ═══════════════════════════════════════════════════════════════════════

!pip install -q xgboost scikit-learn

import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report

# ── Feature set: previous best features + new aircraft-chain features ──
feature_cols_v5 = [
    'Airline', 'Origin', 'Month', 'DayOfWeek', 'ScheduledDepHour',
    'Season', 'IsWeekend', 'Distance',
    'TempC', 'WindSpeedKt', 'WindGustKt', 'VisibSM', 'FlightCategory',
    'AirportHourCongestion', 'RollingDelayRate',
    'PrevLegWasDelayed', 'PrevLegArrDelay',   # ← NEW
]
target_col = 'DepDel15'

model_df_v5 = bts_weather[feature_cols_v5 + [target_col, 'Year']].copy()

cat_cols = ['Airline', 'Origin', 'Season', 'FlightCategory']
for col in cat_cols:
    model_df_v5[col] = model_df_v5[col].astype('category')

# ── Time-based train/test split ──────────────────────────────────────────
train_df_v5 = model_df_v5[model_df_v5['Year'] < 2024]
test_df_v5  = model_df_v5[model_df_v5['Year'] == 2024]

X_train_v5, y_train_v5 = train_df_v5[feature_cols_v5], train_df_v5[target_col]
X_test_v5, y_test_v5   = test_df_v5[feature_cols_v5], test_df_v5[target_col]

print(f'Train set: {len(X_train_v5):,} flights (2022-2023)')
print(f'Test set:  {len(X_test_v5):,} flights (2024)')

scale_pos_weight_v5 = (y_train_v5 == 0).sum() / (y_train_v5 == 1).sum()

# ── Train ──────────────────────────────────────────────────────────────
xgb_model_v5 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight_v5,
    random_state=42,
)
xgb_model_v5.fit(X_train_v5, y_train_v5)

# ── Evaluate ──────────────────────────────────────────────────────────────
y_pred_proba_v5 = xgb_model_v5.predict_proba(X_test_v5)[:, 1]
y_pred_v5 = xgb_model_v5.predict(X_test_v5)

auc_v5 = roc_auc_score(y_test_v5, y_pred_proba_v5)
print(f'\nROC-AUC: {auc_v5:.4f}  (previous best: 0.6903)')
print('\nClassification report:')
print(classification_report(y_test_v5, y_pred_v5, target_names=['On Time', 'Delayed']))

importance_v5 = pd.DataFrame({
    'feature': feature_cols_v5,
    'importance': xgb_model_v5.feature_importances_
}).sort_values('importance', ascending=False)
print('\nFeature importance:')
print(importance_v5.to_string(index=False))

Train set: 3,881,146 flights (2022-2023)
Test set:  2,015,460 flights (2024)

ROC-AUC: 0.7222  (previous best: 0.6903)

Classification report:
              precision    recall  f1-score   support

     On Time       0.87      0.72      0.78   1579892
     Delayed       0.37      0.61      0.46    435568

    accuracy                           0.69   2015460
   macro avg       0.62      0.66      0.62   2015460
weighted avg       0.76      0.69      0.71   2015460


Feature importance:
              feature  importance
      PrevLegArrDelay    0.380989
     ScheduledDepHour    0.176415
     RollingDelayRate    0.125303
               Season    0.046610
              VisibSM    0.041848
              Airline    0.037026
            DayOfWeek    0.033775
             Distance    0.030449
               Origin    0.029628
                TempC    0.020779
    PrevLegWasDelayed    0.016665
                Month    0.015627
           WindGustKt    0.014766
          WindSpeedKt    0.014762

In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Save Final Model Metrics (v5 — with Tail_Number features)
# ═══════════════════════════════════════════════════════════════════════
import json

final_metrics = {
    'model': 'XGBoost',
    'roc_auc': round(auc_v5, 4),
    'train_period': '2022-2023',
    'test_period': '2024',
    'train_size': len(X_train_v5),
    'test_size': len(X_test_v5),
    'features_used': feature_cols_v5,
    'feature_importance': dict(zip(
        feature_cols_v5,
        [round(float(x), 4) for x in xgb_model_v5.feature_importances_]
    )),
    'notes': (
        'Final model. Adding Tail_Number-based aircraft-chain features '
        '(PrevLegArrDelay, PrevLegWasDelayed) improved ROC-AUC from '
        '0.6903 to 0.7222 -- the largest single improvement across all '
        'iterations. PrevLegArrDelay (continuous prior-leg delay in '
        'minutes) was the single most important feature (0.381), '
        'far exceeding PrevLegWasDelayed (binary flag, 0.017) -- '
        'delay MAGNITUDE carries far more signal than a simple '
        'threshold flag. Known limitation: dataset is scoped to '
        'flights touching the 10 hub airports, so an aircraft\'s '
        '"previous leg" in this data is not always its true '
        'immediately-prior flight in the full national network.'
    )
}

metrics_path = DIR_PROCESSED / 'model_metrics_final.json'
with open(metrics_path, 'w') as f:
    json.dump(final_metrics, f, indent=2)

hf_api.upload_file(
    path_or_fileobj=str(metrics_path),
    path_in_repo="model_metrics_final.json",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)

print(f'Saved and uploaded: {metrics_path}')
print(json.dumps(final_metrics, indent=2))

Saved and uploaded: /content/airline-disruption/data/processed/model_metrics_final.json
{
  "model": "XGBoost",
  "roc_auc": 0.7222,
  "train_period": "2022-2023",
  "test_period": "2024",
  "train_size": 3881146,
  "test_size": 2015460,
  "features_used": [
    "Airline",
    "Origin",
    "Month",
    "DayOfWeek",
    "ScheduledDepHour",
    "Season",
    "IsWeekend",
    "Distance",
    "TempC",
    "WindSpeedKt",
    "WindGustKt",
    "VisibSM",
    "FlightCategory",
    "AirportHourCongestion",
    "RollingDelayRate",
    "PrevLegWasDelayed",
    "PrevLegArrDelay"
  ],
  "feature_importance": {
    "Airline": 0.037,
    "Origin": 0.0296,
    "Month": 0.0156,
    "DayOfWeek": 0.0338,
    "ScheduledDepHour": 0.1764,
    "Season": 0.0466,
    "IsWeekend": 0.0,
    "Distance": 0.0304,
    "TempC": 0.0208,
    "WindSpeedKt": 0.0148,
    "WindGustKt": 0.0148,
    "VisibSM": 0.0418,
    "FlightCategory": 0.0067,
    "AirportHourCongestion": 0.0086,
    "RollingDelayRate": 0.1253,
    "Pr

In [18]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Build Polished KPI Dashboard (v8 — Refined Typography, Spacing, Color)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Applies six targeted refinements:
#   1. KPI cards left-aligned (number + label), subtext to Sentence case
#   2. Monthly trend x-axis switched to true date type, horizontal ticks,
#      every 6 months (no clipping/overlap)
#   3. Tighter vertical gap between header and KPI row
#   4. Uniform spacing between all 4 chart panels
#   5. Trend line color unified to navy (was purple); heatmap cells
#      given a thin gap (xgap/ygap) so cells don't visually crowd axis labels
#   6. Cancellation Rate KPI number given a distinct alert-red color;
#      all other KPIs stay navy
# ═══════════════════════════════════════════════════════════════════════

import plotly.graph_objects as go
from plotly.subplots import make_subplots

NAVY          = '#1e3a8a'
ALERT_RED     = '#dc2626'   # reserved strictly for Cancellation Rate
PAGE_BG       = '#f1f5f9'
CARD_BG       = '#ffffff'
GRID_COLOR    = '#e2e8f0'
TEXT_MUTED    = '#64748b'
TEXT_DARK     = '#0f172a'
FONT_FAMILY   = 'Helvetica Neue, Arial, sans-serif'

# ── Pre-aggregate ─────────────────────────────────────────────────────
severity_counts = bts_weather['SeverityTier'].value_counts().reindex(
    ['On Time', 'Minor', 'Significant', 'Severe', 'Cancelled']
)
airport_delay_rate = bts_weather.groupby('Origin', observed=True)['DepDel15'].mean().sort_values(ascending=False) * 100

monthly_trend = bts_weather.groupby(
    bts_weather['FlightDate'].dt.to_period('M'), observed=True
)['DepDel15'].mean() * 100
monthly_trend.index = monthly_trend.index.to_timestamp()   # ← real dates, not strings

hourly_stats = bts_weather.groupby(['Origin', 'ScheduledDepHour'], observed=True)['DepDel15'].agg(['mean', 'count'])
hourly_stats['mean'] = hourly_stats['mean'] * 100
MIN_FLIGHTS_PER_CELL = 500
hourly_stats.loc[hourly_stats['count'] < MIN_FLIGHTS_PER_CELL, 'mean'] = np.nan
hourly_by_airport = hourly_stats['mean'].unstack('ScheduledDepHour')

total_flights = len(bts_weather)
overall_delay_rate = bts_weather['DepDel15'].mean() * 100
cancel_rate = bts_weather['Cancelled'].mean() * 100
cascade_rate = bts_weather['IsCascadeRisk'].mean() * 100

# ── Chart grid — uniform spacing between all 4 panels ────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Severity Breakdown', 'Delay Rate by Airport (avg.)',
        'Monthly Delay Rate Trend (avg.)',
        f'Delay Rate by Hour × Airport (avg., min {MIN_FLIGHTS_PER_CELL} flights/cell)'
    ),
    vertical_spacing=0.14, horizontal_spacing=0.12   # ← equalized
)

fig.add_trace(go.Bar(
    x=severity_counts.index, y=severity_counts.values,
    marker_color=NAVY, marker_line_width=0,
    text=[f'{v:,.0f}' for v in severity_counts.values], textposition='outside',
    textfont=dict(size=11, color=TEXT_MUTED),
), row=1, col=1)
fig.update_yaxes(title_text='Number of Flights', row=1, col=1)

fig.add_trace(go.Bar(
    x=airport_delay_rate.index, y=airport_delay_rate.values,
    marker_color=NAVY, marker_line_width=0,
    text=[f'{v:.1f}%' for v in airport_delay_rate.values], textposition='outside',
    textfont=dict(size=11, color=TEXT_MUTED),
), row=1, col=2)
fig.update_yaxes(title_text='Avg. Delay Rate (%)', row=1, col=2)

# Trend line: unified to navy (was purple)
fig.add_trace(go.Scatter(
    x=monthly_trend.index, y=monthly_trend.values,
    mode='lines+markers',
    line=dict(color=NAVY, width=3, shape='spline'),
    marker=dict(size=6, color=NAVY, line=dict(color='white', width=1)),
    fill='tozeroy', fillcolor='rgba(30,58,138,0.08)',
), row=2, col=1)
fig.update_yaxes(title_text='Avg. Delay Rate (%)', row=2, col=1)
fig.update_xaxes(
    title_text='Month', tickangle=0, tickformat='%b %Y',
    dtick='M6', row=2, col=1
)

# Heatmap: xgap/ygap adds thin separation between cells, cleaning up
# how the leftmost column reads against the y-axis labels
fig.add_trace(go.Heatmap(
    z=hourly_by_airport.values, x=hourly_by_airport.columns,
    y=hourly_by_airport.index, colorscale='YlOrRd',
    xgap=1.5, ygap=1.5,
    colorbar=dict(title='Avg. Delay<br>Rate (%)', len=0.4, y=0.19, thickness=14),
    hovertemplate='Airport: %{y}<br>Hour: %{x}:00<br>Avg. Delay Rate: %{z:.1f}%<extra></extra>',
), row=2, col=2)
fig.update_xaxes(title_text='Scheduled Departure Hour (24hr)', row=2, col=2)
fig.update_yaxes(title_text='Origin Airport', automargin=True, row=2, col=2)

fig.update_xaxes(showgrid=False, zeroline=False, showline=True, linecolor=GRID_COLOR)
fig.update_yaxes(showgrid=True, gridcolor=GRID_COLOR, zeroline=False, showline=False)

# ── Title ──────────────────────────────────────────────────────────────
fig.add_annotation(
    x=0, y=1.36, xref='paper', yref='paper', xanchor='left', yanchor='top',
    text=f"<b style='font-size:24px;color:{TEXT_DARK};letter-spacing:0.3px'>AIRLINE DISRUPTION DETECTION</b>"
         f"<br><span style='font-size:14px;color:{NAVY};font-weight:600'>Phase 1 — Operations Dashboard</span>"
         f"<br><span style='font-size:11.5px;color:{TEXT_MUTED}'>10 U.S. hub airports · 2022–2024 · "
         f"{total_flights:,} departures analyzed</span>",
    showarrow=False, align='left', font=dict(family=FONT_FAMILY),
)

# ── KPI cards: left-aligned, Sentence case, Cancellation Rate in red ────
kpi_specs = [
    (f'{total_flights:,}', 'Total flights', '(count)', 0.02, 0.24, NAVY),
    (f'{overall_delay_rate:.1f}%', 'Delay rate (≥15 min)', '(avg. across all flights)', 0.27, 0.49, NAVY),
    (f'{cancel_rate:.1f}%', 'Cancellation rate', '(avg. across all flights)', 0.52, 0.74, ALERT_RED),
    (f'{cascade_rate:.1f}%', 'Cascade-risk flights', '(avg. across all flights)', 0.77, 0.99, NAVY),
]

shapes = [
    dict(type='line', xref='paper', yref='paper', x0=0, x1=1, y0=1.24, y1=1.24,
         line=dict(color=NAVY, width=2)),
]
LEFT_PAD = 0.015
for value_str, label, sublabel, x0, x1, number_color in kpi_specs:
    shapes.append(dict(
        type='rect', xref='paper', yref='paper',
        x0=x0, x1=x1, y0=1.04, y1=1.22,
        fillcolor=CARD_BG, line=dict(color=GRID_COLOR, width=1), layer='below',
    ))
    shapes.append(dict(
        type='rect', xref='paper', yref='paper',
        x0=x0, x1=x1, y0=1.215, y1=1.22,
        fillcolor=NAVY, line=dict(width=0), layer='below',   # top accent bar stays navy for all
    ))
    fig.add_annotation(
        x=x0 + LEFT_PAD, y=1.13, xref='paper', yref='paper',
        xanchor='left', yanchor='middle', align='left',
        text=f"<b style='font-size:26px;color:{number_color}'>{value_str}</b>"
             f"<br><span style='font-size:10.5px;color:{TEXT_DARK}'>{label}</span>"
             f"<br><span style='font-size:8.5px;color:{TEXT_MUTED}'>{sublabel}</span>",
        showarrow=False, font=dict(family=FONT_FAMILY),
    )

fig.update_layout(
    template='plotly_white',
    height=1060, width=1150,
    font=dict(family=FONT_FAMILY, size=12, color='#1e293b'),
    showlegend=False,
    plot_bgcolor=CARD_BG,
    paper_bgcolor=PAGE_BG,
    margin=dict(t=290, l=60, r=60, b=60),
    shapes=shapes,
)
fig.update_annotations(font_size=13, selector=dict(font=dict(family=FONT_FAMILY)))

fig.show()

dashboard_path = DIR_PROCESSED / 'phase1_dashboard.html'
fig.write_html(str(dashboard_path), include_plotlyjs='cdn')
print(f'\nDashboard exported to: {dashboard_path}')


Dashboard exported to: /content/airline-disruption/data/processed/phase1_dashboard.html


In [19]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Upload Final Dashboard to Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
hf_api.upload_file(
    path_or_fileobj=str(dashboard_path),
    path_in_repo="phase1_dashboard.html",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print("✅ phase1_dashboard.html uploaded to Hugging Face Hub")

✅ phase1_dashboard.html uploaded to Hugging Face Hub


In [20]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Download the Dashboard HTML File
# ═══════════════════════════════════════════════════════════════════════
from google.colab import files
files.download(str(dashboard_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>